In [1]:
# conda env 'at3d_pure'

from collections import OrderedDict
import at3d
import xarray as xr
import numpy as np

In [2]:
wavelength = 0.67

rte_grid_title = 'default'

aerosol_title = 'MISR_MIX21_AOD0p200_LO20'
aerosol_file = '../data/AEROSOL_{}_RTG_{}.nc'.format(aerosol_title, rte_grid_title)
aerosol = xr.load_dataset(aerosol_file)

atmos_title = 'CAMP2Ex_dropsonde_20191002022856'
atmos_file = '../data/ATMOS_{}_RTG_{}.nc'.format(atmos_title, rte_grid_title)
atmos = xr.load_dataset(atmos_file)

assert np.array_equal(aerosol.x.data, atmos.x.data)
assert np.array_equal(aerosol.y.data, atmos.y.data)
assert np.array_equal(aerosol.z.data, atmos.z.data)

out_file = '../data/BACKGROUND_{}_{}_RTG_{}.nc'.format(aerosol_title, atmos_title, rte_grid_title)

In [4]:
background_optical_scatterers = OrderedDict()
merged_properties = at3d.medium.mix_optical_properties(
    atmos,
    aerosol,
    asymmetry_tol=0.02,
    maxnphase=18
)
merged_properties.to_netcdf(out_file)

In [5]:
merged_properties

<xarray.Dataset>
Dimensions:        (stokes_index: 6, legendre_index: 21, table_index: 17,
                    num_micro: 1, x: 65, y: 65, z: 17)
Coordinates:
    wavelength     float64 0.67
  * x              (x) float64 0.0 0.275 0.55 0.825 ... 16.78 17.05 17.33 17.6
  * y              (y) float64 0.0 0.275 0.55 0.825 ... 16.78 17.05 17.33 17.6
  * z              (z) float64 0.0 0.2 0.39 0.4 0.5 0.6 ... 1.3 1.4 1.41 1.6 1.8
    table_index    (num_micro, x, y, z) int32 1 2 3 4 5 6 ... 12 13 14 15 16 17
Dimensions without coordinates: stokes_index, legendre_index, num_micro
Data variables:
    legcoef        (stokes_index, legendre_index, table_index) float32 1.0 .....
    phase_weights  (num_micro, x, y, z) float32 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0
    extinction     (x, y, z) float64 0.1056 0.09593 0.08759 ... 0.04946 0.9467
    ssalb          (x, y, z) float64 1.0 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0 1.0 1.0
    mixture        bool True
    delx           float64 0.275
    dely           float64 0.275
Attributes:
    asymmetry_tolerance:            0.02
    phase_tolerance:                0.01
    linear_polarization_tolerance:  0.01
    description:                    This set of optical properties is a mixtu...

In [10]:
np.min(merged_properties.ssalb.data)

0.9999999312076963